# Tổng hợp Q/K/V Attention từ checkpoint Transformer

Notebook này nối lại ba ý tưởng: Positional Encoding, Scaled Dot-Product Attention và Multi-Head Attention. Khác các demo tối giản, mọi tensor `Q`, `K`, `V` trong phần attention đều được chiếu bằng trọng số thật từ `mini_transformer_best_bleu.pt`, không sinh ngẫu nhiên.

## 1. Imports và cấu hình

`torch.manual_seed(42)` chỉ dùng cho phần minh họa Positional Encoding nhỏ. Các phần attention phía sau lấy embedding và projection weight từ checkpoint.

In [ ]:
import html
import json
import math
import pickle
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
try:
    import seaborn as sns
except ModuleNotFoundError:
    class _SimpleSeaborn:
        def heatmap(self, data, annot=False, fmt=".2f", cmap=None, xticklabels=None, yticklabels=None, square=False, cbar=True, ax=None, **kwargs):
            ax = ax or plt.gca()
            data = np.asarray(data)
            image = ax.imshow(data, cmap=cmap, aspect="equal" if square else "auto")
            if cbar:
                plt.colorbar(image, ax=ax)
            if xticklabels is not None:
                ax.set_xticks(np.arange(data.shape[1]))
                ax.set_xticklabels(xticklabels)
            if yticklabels is not None:
                ax.set_yticks(np.arange(data.shape[0]))
                ax.set_yticklabels(yticklabels)
            if annot:
                for row in range(data.shape[0]):
                    for col in range(data.shape[1]):
                        ax.text(col, row, format(data[row, col], fmt), ha="center", va="center", fontsize=8)
            return image

    sns = _SimpleSeaborn()
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer, normalizers
from tokenizers.models import BPE
from tokenizers.normalizers import Lowercase, NFKC
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

# Cố định seed để phần minh họa positional encoding có thể tái lập.
torch.manual_seed(42)

CKPT_PATH = "../model/mini_transformer_best_bleu.pt"
VOCAB_PATH = "../model/shared_vocab_recreated.pkl"
TOKENIZER_JSON_PATH = "../model/tokenizer.json"
LAYER_PREFIX = "encoder.layers.0.self_attn"
NORM_PREFIX = "encoder.layers.0.norm1"


def safe_text(value):
    text = str(value)
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    try:
        text.encode(encoding)
        return text
    except UnicodeEncodeError:
        return text.encode(encoding, "backslashreplace").decode(encoding)

## 2. Positional Encoding từ scratch

Công thức giữ nguyên tinh thần Transformer gốc: vị trí chẵn dùng `sin`, vị trí lẻ dùng `cos`. PE không học bằng gradient; nó được cộng vào embedding để token cùng loại nhưng ở vị trí khác nhau có biểu diễn khác nhau.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()

        # pe lưu vector vị trí cho mọi position tới max_len, không học bằng gradient.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        # div_term điều khiển tần số sin/cos khác nhau cho từng cặp chiều.
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # Chiều chẵn dùng sin, chiều lẻ dùng cos đúng công thức Transformer gốc.
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        # register_buffer đưa PE đi cùng module nhưng không xem là trainable parameter.
        self.register_buffer("pe", pe)

    def forward(self, x):
        # Cộng positional encoding theo độ dài sequence hiện tại.
        return x + self.pe[:, : x.size(1)]

In [ ]:
demo_vocab = {"chó": 0, "đuổi": 1, "mèo": 2}
demo_d_model = 16
demo_seq_len = 3
s1_words = ["Chó", "đuổi", "mèo"]
s2_words = ["Mèo", "đuổi", "chó"]
s1_idx = torch.tensor([demo_vocab["chó"], demo_vocab["đuổi"], demo_vocab["mèo"]])
s2_idx = torch.tensor([demo_vocab["mèo"], demo_vocab["đuổi"], demo_vocab["chó"]])

# Embedding demo là ngẫu nhiên có seed cố định; phần checkpoint phía trên không dùng embedding này.
demo_embedding = nn.Embedding(num_embeddings=len(demo_vocab), embedding_dim=demo_d_model)
demo_pe = PositionalEncoding(demo_d_model, max_len=demo_seq_len)

emb1 = demo_embedding(s1_idx).unsqueeze(0)
emb2 = demo_embedding(s2_idx).unsqueeze(0)
final1 = demo_pe(emb1).squeeze(0)
final2 = demo_pe(emb2).squeeze(0)

scale = math.sqrt(demo_d_model)
# So sánh attention score trước và sau khi cộng positional encoding.
attn1_no_pe = (emb1.squeeze(0) @ emb1.squeeze(0).T / scale).detach().numpy()
attn2_no_pe = (emb2.squeeze(0) @ emb2.squeeze(0).T / scale).detach().numpy()
attn1_with_pe = (final1 @ final1.T / scale).detach().numpy()
attn2_with_pe = (final2 @ final2.T / scale).detach().numpy()

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
sns.heatmap(attn1_no_pe, ax=axes[0, 0], annot=True, fmt=".2f", cmap="Greys", xticklabels=s1_words, yticklabels=s1_words, cbar=False)
axes[0, 0].set_title("Không PE: Chó đuổi mèo")
sns.heatmap(attn2_no_pe, ax=axes[0, 1], annot=True, fmt=".2f", cmap="Greys", xticklabels=s2_words, yticklabels=s2_words, cbar=False)
axes[0, 1].set_title("Không PE: Mèo đuổi chó")
sns.heatmap(attn1_with_pe, ax=axes[1, 0], annot=True, fmt=".2f", cmap="Reds", xticklabels=s1_words, yticklabels=s1_words, cbar=False)
axes[1, 0].set_title("Có PE: Chó đuổi mèo")
sns.heatmap(attn2_with_pe, ax=axes[1, 1], annot=True, fmt=".2f", cmap="Reds", xticklabels=s2_words, yticklabels=s2_words, cbar=False)
axes[1, 1].set_title("Có PE: Mèo đuổi chó")
plt.tight_layout()
plt.show()

x_axis = np.linspace(0, 6, 1000)
num_pairs = 4
fig, axes = plt.subplots(
    nrows=num_pairs,
    ncols=2,
    figsize=(14, 10),
    sharex=True,
)

positions = np.arange(demo_seq_len)
for i in range(num_pairs):
    # Mỗi cặp chiều sin/cos có một tần số riêng.
    scale_i = 10000 ** ((2 * i) / demo_d_model)

    y_sin = np.sin(x_axis / scale_i)
    axes[i, 0].plot(x_axis, y_sin, linewidth=2)
    sin_points = np.sin(positions / scale_i)
    axes[i, 0].scatter(positions, sin_points, s=200, zorder=10)

    for p, y in zip(positions, sin_points):
        axes[i, 0].annotate(
            f"pos={p}",
            (p, y),
            fontsize=10,
            fontweight="bold",
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
        )

    axes[i, 0].set_xlim(-0.5, 3.5)
    axes[i, 0].set_title(
        rf"Dim {2 * i}: "
        rf"$\sin\left(\frac{{pos}}{{10000^{{{2 * i}/{demo_d_model}}}}}\right)$",
        pad=25,
    )
    axes[i, 0].grid(alpha=0.3)

    y_cos = np.cos(x_axis / scale_i)
    axes[i, 1].plot(x_axis, y_cos, linewidth=2)
    cos_points = np.cos(positions / scale_i)
    axes[i, 1].scatter(positions, cos_points, s=200, zorder=10)

    for p, y in zip(positions, cos_points):
        axes[i, 1].annotate(
            f"pos={p}",
            (p, y),
            fontsize=10,
            fontweight="bold",
            textcoords="offset points",
            xytext=(0, 12),
            ha="center",
        )

    axes[i, 1].set_xlim(-0.5, 3.5)
    axes[i, 1].set_title(
        rf"Dim {2 * i + 1}: "
        rf"$\cos\left(\frac{{pos}}{{10000^{{{2 * i}/{demo_d_model}}}}}\right)$",
        pad=25,
    )
    axes[i, 1].grid(alpha=0.3)

axes[-1, 0].set_xlabel("Position")
axes[-1, 1].set_xlabel("Position")

fig.suptitle(
    "Positional Encoding Functions and Sampled PE Values",
    fontsize=16,
)

plt.tight_layout()
plt.show()

## 3. Load vocab và checkpoint

Pickle cần thấy lại class `BPETokenizer` trước khi `pickle.load`, nên ta định nghĩa lại class và `clean_text` giống notebook trích xuất Q/K/V.

In [ ]:
def clean_text(text):
    # Chuẩn hóa text giống lúc train tokenizer để id token khớp checkpoint.
    text = html.unescape(text)
    text = text.strip().lower()
    text = re.sub(r"[“”“«»]", '"', text)
    text = re.sub(r"[‘’`´]", "'", text)
    text = re.sub(r"[–—]", "-", text)
    return text


class BPETokenizer:
    def __init__(self, vocab_size=18000):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(BPE(unk_token="<unk>"))
        self.tokenizer.normalizer = normalizers.Sequence([NFKC(), Lowercase()])
        self.tokenizer.pre_tokenizer = Whitespace()
        self.trainer = BpeTrainer(
            special_tokens=["<pad>", "<sos>", "<eos>", "<unk>"],
            vocab_size=self.vocab_size,
        )

    def train(self, sentences):
        cleaned = [clean_text(s) for s in sentences]
        self.tokenizer.train_from_iterator(cleaned, self.trainer)

    def encode(self, sentence, add_special=True):
        cleaned = clean_text(sentence)
        ids = self.tokenizer.encode(cleaned).ids
        if add_special:
            ids = [1] + ids + [2]
        return ids

    def decode(self, ids):
        filtered = [idx for idx in ids if idx not in [0, 1, 2]]
        return self.tokenizer.decode(filtered)

    @property
    def word2idx(self):
        return {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}

    @property
    def idx2word(self):
        class _IdToToken:
            def __init__(self, tok):
                self.tok = tok

            def __getitem__(self, idx):
                val = self.tok.id_to_token(idx)
                return val if val is not None else "<unk>"

            def get(self, idx, default="<unk>"):
                val = self.tok.id_to_token(idx)
                return val if val is not None else default

        return _IdToToken(self.tokenizer)

    def __len__(self):
        return self.tokenizer.get_vocab_size()


# Cho pickle biết BPETokenizer/clean_text thuộc __main__ giống notebook gốc đã lưu vocab.
sys.modules["__main__"].BPETokenizer = BPETokenizer
sys.modules["__main__"].clean_text = clean_text


class JsonBPETokenizer:
    def __init__(self, vocab, merges, unk_token="<unk>"):
        self.vocab = vocab
        self.id_to_piece = {idx: piece for piece, idx in vocab.items()}
        self.merge_ranks = {tuple(pair): rank for rank, pair in enumerate(merges)}
        self.unk_id = vocab.get(unk_token, 3)

    @classmethod
    def from_file(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        model = data["model"]
        return cls(model["vocab"], model["merges"], model.get("unk_token", "<unk>"))

    def _pre_tokenize(self, sentence):
        return re.findall(r"\w+|[^\w\s]", clean_text(sentence), flags=re.UNICODE)

    def _bpe_word(self, word):
        pieces = list(word)
        if not pieces:
            return []

        while len(pieces) > 1:
            pairs = [(pieces[i], pieces[i + 1]) for i in range(len(pieces) - 1)]
            ranked = [(self.merge_ranks[pair], i, pair) for i, pair in enumerate(pairs) if pair in self.merge_ranks]
            if not ranked:
                break

            _, merge_at, pair = min(ranked)
            merged = pair[0] + pair[1]
            pieces = pieces[:merge_at] + [merged] + pieces[merge_at + 2 :]

        return pieces

    def encode(self, sentence, add_special=True):
        ids = []
        for word in self._pre_tokenize(sentence):
            ids.extend(self.vocab.get(piece, self.unk_id) for piece in self._bpe_word(word))
        if add_special:
            ids = [1] + ids + [2]
        return ids

    def decode(self, ids):
        return " ".join(self.id_to_piece.get(int(idx), "<unk>") for idx in ids if int(idx) not in [0, 1, 2])

    @property
    def idx2word(self):
        class _IdToPiece:
            def __init__(self, id_to_piece):
                self.id_to_piece = id_to_piece

            def __getitem__(self, idx):
                return self.id_to_piece.get(int(idx), "<unk>")

            def get(self, idx, default="<unk>"):
                return self.id_to_piece.get(int(idx), default)

        return _IdToPiece(self.id_to_piece)

    def __len__(self):
        return len(self.vocab)

In [ ]:
try:
    # Ưu tiên vocab pickle vì đây là artifact được checkpoint dùng trực tiếp.
    with open(VOCAB_PATH, "rb") as f:
        shared_vocab = pickle.load(f)
    print("Loaded vocab from pickle:", VOCAB_PATH)
except Exception as exc:
    print("Could not load pickle vocab; using tokenizer.json instead:", safe_text(exc))
    try:
        shared_vocab = BPETokenizer()
        shared_vocab.tokenizer = Tokenizer.from_file(TOKENIZER_JSON_PATH)
    except Exception as json_exc:
        # Fallback cuối cùng chỉ đọc JSON BPE để notebook vẫn chạy khi thiếu extension tokenizers.
        print("Could not load tokenizer.json with tokenizers; using local JSON BPE:", safe_text(json_exc))
        shared_vocab = JsonBPETokenizer.from_file(TOKENIZER_JSON_PATH)

# Load checkpoint trên CPU để notebook không phụ thuộc GPU.
state_dict = torch.load(CKPT_PATH, map_location="cpu")

vocab_size, d_model = state_dict["encoder.embedding.weight"].shape
num_heads = 8
# d_k là độ rộng vector của mỗi head sau khi tách d_model.
d_k = d_model // num_heads

assert len(shared_vocab) == vocab_size
assert state_dict["encoder.embedding.weight"].shape == torch.Size([18000, 512])
assert state_dict[f"{LAYER_PREFIX}.W_q.weight"].shape == torch.Size([512, 512])
assert state_dict[f"{LAYER_PREFIX}.W_k.weight"].shape == torch.Size([512, 512])
assert state_dict[f"{LAYER_PREFIX}.W_v.weight"].shape == torch.Size([512, 512])
assert d_model == 512 and num_heads == 8 and d_k == 64

print("Vocab size:", len(shared_vocab))
print("Checkpoint vocab_size:", vocab_size)
print("d_model:", d_model)
print("num_heads:", num_heads)
print("d_k:", d_k)

In [ ]:
def ids_to_tokens(vocab, ids):
    tokens = []
    for idx in ids:
        idx = int(idx)
        # Hỗ trợ nhiều kiểu vocab khác nhau nhưng luôn trả về token string theo id.
        if hasattr(vocab, "idx2word"):
            tokens.append(vocab.idx2word[idx])
        elif hasattr(vocab, "idx_to_token"):
            tokens.append(vocab.idx_to_token[idx])
        elif hasattr(vocab, "id_to_token"):
            tokens.append(vocab.id_to_token(idx))
        elif hasattr(vocab, "itos"):
            tokens.append(vocab.itos[idx])
        else:
            tokens.append(f"id_{idx}")
    return tokens


src_sentence = "tôi muốn học transformer"
# add_special=False để heatmap chỉ hiển thị token nội dung, không thêm <sos>/<eos>.
src_ids = shared_vocab.encode(src_sentence, add_special=False)
src_tokens = ids_to_tokens(shared_vocab, src_ids)
src = torch.tensor([src_ids], dtype=torch.long)

print("sentence:", safe_text(src_sentence))
print("src_ids:", src_ids)
print("src_tokens:", safe_text(src_tokens))
print("src shape:", tuple(src.shape))

## 4. Tạo input encoder thật

Ta không cần dựng full Transformer. Chỉ cần làm đúng chuỗi tính toán đi vào self-attention layer đầu tiên: embedding thật, cộng PE, rồi layer norm `norm1` của `encoder.layers.0`.

In [ ]:
with torch.no_grad():
    # Embedding thật từ checkpoint, scale sqrt(d_model) giống Transformer.
    x = F.embedding(src, state_dict["encoder.embedding.weight"]) * math.sqrt(d_model)
    # Cộng positional encoding trước khi đi vào encoder layer.
    x = PositionalEncoding(d_model)(x)
    # Dùng đúng norm1 của encoder.layers.0 để tạo input self-attention thật.
    norm_x = F.layer_norm(
        x,
        [d_model],
        weight=state_dict[f"{NORM_PREFIX}.weight"],
        bias=state_dict[f"{NORM_PREFIX}.bias"],
    )

print("x:", tuple(x.shape))
print("norm_x:", tuple(norm_x.shape))

## 5. Trích xuất và dùng Q/K/V weight thật

`W_q`, `W_k`, `W_v` biến cùng một `norm_x` thành ba không gian khác nhau. Đây là Q/K/V thật của checkpoint cho câu mẫu, không phải tensor random.

In [ ]:
def linear_from_checkpoint(state_dict, prefix):
    layer = nn.Linear(d_model, d_model)
    with torch.no_grad():
        # Copy weight/bias từ checkpoint vào layer torch.nn.Linear tự dựng.
        layer.weight.copy_(state_dict[f"{prefix}.weight"])
        layer.bias.copy_(state_dict[f"{prefix}.bias"])
    layer.eval()
    return layer


W_q = linear_from_checkpoint(state_dict, f"{LAYER_PREFIX}.W_q")
W_k = linear_from_checkpoint(state_dict, f"{LAYER_PREFIX}.W_k")
W_v = linear_from_checkpoint(state_dict, f"{LAYER_PREFIX}.W_v")
W_o = linear_from_checkpoint(state_dict, f"{LAYER_PREFIX}.W_o")

with torch.no_grad():
    # Cùng norm_x được chiếu sang ba vai trò khác nhau: query, key, value.
    Q_linear = W_q(norm_x)
    K_linear = W_k(norm_x)
    V_linear = W_v(norm_x)

print("Q_linear:", tuple(Q_linear.shape))
print("K_linear:", tuple(K_linear.shape))
print("V_linear:", tuple(V_linear.shape))

## 6. ScaleDotAttention

Nhóm này gồm phép tính thủ công, class `ScaledDotProductAttention`, và visualization của chính class đó. Mỗi dòng heatmap là một query token đang nhìn sang toàn bộ key tokens.

In [ ]:
batch_size, seq_len = src.shape

with torch.no_grad():
    # Đưa Q/K/V về dạng [batch, head, token, d_k] để mỗi head tính attention riêng.
    Q = Q_linear.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K_linear.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    V = V_linear.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)

    # raw_scores là Q @ K^T: độ khớp giữa từng query token và từng key token.
    raw_scores = torch.matmul(Q, K.transpose(-2, -1))
    # Chia sqrt(d_k) để score không quá lớn trước softmax.
    scaled_scores = raw_scores / math.sqrt(d_k)
    # Softmax trên chiều key để mỗi query có một phân phối attention.
    attention_weights = torch.softmax(scaled_scores, dim=-1)
    # Context là tổng có trọng số của V theo attention_weights.
    context = torch.matmul(attention_weights, V)

assert Q.shape == K.shape == V.shape == torch.Size([1, num_heads, seq_len, d_k])
assert torch.allclose(attention_weights.sum(dim=-1), torch.ones_like(attention_weights.sum(dim=-1)), atol=1e-6)

print("Q:", tuple(Q.shape))
print("K:", tuple(K.shape))
print("V:", tuple(V.shape))
print("raw_scores:", tuple(raw_scores.shape))
print("scaled_scores:", tuple(scaled_scores.shape))
print("attention_weights:", tuple(attention_weights.shape))
print("context:", tuple(context.shape))

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, masked=None):
        # d_k là độ rộng của mỗi head, dùng làm mẫu số scale.
        d_k = Q.size(-1)
        # Tính attention score đã scale cho mọi query-key pair.
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if masked is not None:
            # Mask chặn những vị trí không được attend bằng score rất âm.
            scores = scores.masked_fill(masked == 0, -1e4)

        # Softmax biến score thành xác suất attention theo chiều key.
        attn_weights = torch.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        # Nhân với V để thu được context vector cho từng query.
        context = torch.matmul(attn_weights, V)

        return context, attn_weights

In [ ]:
sdp_attention = ScaledDotProductAttention(dropout=0.0)
sdp_attention.eval()

with torch.no_grad():
    # Visualization dùng output của class ScaledDotProductAttention, không dùng trực tiếp biến thủ công.
    sdp_context, sdp_attention_weights = sdp_attention(Q, K, V)

# Kiểm tra class khớp phép tính thủ công phía trên.
assert torch.allclose(sdp_attention_weights, attention_weights, atol=1e-6)
assert torch.allclose(sdp_context, context, atol=1e-6)

head_id = 0
attn_map = sdp_attention_weights[0, head_id].detach().cpu().numpy()

plt.figure(figsize=(8, 6))
sns.heatmap(
    attn_map,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=src_tokens,
    yticklabels=src_tokens,
    square=True,
    cbar=True,
)
plt.title(f"Scaled Dot-Product Attention thật\nEncoder Layer 0 - Head {head_id}", fontsize=13, pad=15, fontweight="bold")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. MultiHeadAttention

Nhóm này gồm class `MultiHeadAttention`, bước nạp weight từ checkpoint, kiểm tra khớp với phép tính thủ công, và visualization của output attention từ module multi-head.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model phải chia hết cho num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        # Mỗi head nhận một lát d_k của không gian d_model.
        self.d_k = d_model // num_heads

        # Bốn linear layer tương ứng W_q, W_k, W_v, W_o trong checkpoint.
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.attention = ScaledDotProductAttention(dropout)

    def load_qkv_from_checkpoint(self, state_dict, layer_prefix):
        with torch.no_grad():
            for name in ["W_q", "W_k", "W_v", "W_o"]:
                # Copy đúng tensor weight/bias theo tên layer trong checkpoint.
                layer = getattr(self, name)
                layer.weight.copy_(state_dict[f"{layer_prefix}.{name}.weight"])
                layer.bias.copy_(state_dict[f"{layer_prefix}.{name}.bias"])
        return self

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)

        # Linear projection rồi reshape từ [batch, token, d_model] sang [batch, head, token, d_k].
        Q = self.W_q(q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(k).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(v).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # ScaledDotProductAttention chạy song song trên toàn bộ heads.
        context, attn_weights = self.attention(Q, K, V, mask)
        # Concat các head về lại [batch, token, d_model].
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        # W_o trộn thông tin sau khi concat multi-head context.
        output = self.W_o(context)

        return output, attn_weights, context

In [ ]:
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.0)
# Nạp weight thật từ encoder layer 0 self-attention.
mha.load_qkv_from_checkpoint(state_dict, LAYER_PREFIX)
mha.eval()

with torch.no_grad():
    # Encoder self-attention dùng cùng norm_x cho q, k, v.
    mha_output, mha_attention_weights, mha_context = mha(norm_x, norm_x, norm_x)
    # Ghép context thủ công để đối chiếu với phần concat trong MultiHeadAttention.
    manual_context_concat = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    manual_output = W_o(manual_context_concat)

# Ba assert kiểm tra module MHA khớp phép tính thủ công ở attention, context và output.
assert torch.allclose(mha_attention_weights, attention_weights, atol=1e-6)
assert torch.allclose(mha_context, manual_context_concat, atol=1e-6)
assert torch.allclose(mha_output, manual_output, atol=1e-6)

print("mha_output:", tuple(mha_output.shape))
print("mha_attention_weights:", tuple(mha_attention_weights.shape))
print("MHA output matches the manual calculation.")

In [ ]:
# Lấy attention từ module MultiHeadAttention: [head, query_token, key_token].
attn_matrices = mha_attention_weights[0].detach().cpu().numpy()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.reshape(-1)
for h in range(num_heads):
    # Mỗi subplot là một head với pattern attention riêng.
    sns.heatmap(
        attn_matrices[h],
        annot=True,
        fmt=".2f",
        cmap="viridis",
        xticklabels=src_tokens,
        yticklabels=src_tokens,
        square=True,
        cbar=False,
        ax=axes[h],
    )
    axes[h].set_title(f"Head {h}", fontsize=11, fontweight="bold")
    axes[h].set_xlabel("Keys")
    axes[h].set_ylabel("Queries" if h % 4 == 0 else "")
    axes[h].set_xticklabels(src_tokens, rotation=45, ha="right")
    axes[h].set_yticklabels(src_tokens, rotation=0)

fig.suptitle("Attention thật của 8 heads - Encoder Layer 0", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Trung bình theo head để xem pattern attention chung của MultiHeadAttention.
mean_attn = attn_matrices.mean(axis=0)

plt.figure(figsize=(8, 6))
sns.heatmap(
    mean_attn,
    annot=True,
    fmt=".2f",
    cmap="magma",
    xticklabels=src_tokens,
    yticklabels=src_tokens,
    square=True,
)
plt.title("Attention trung bình của 8 heads", fontsize=13, pad=15, fontweight="bold")
plt.xlabel("Keys")
plt.ylabel("Queries")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()